In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: Una Qiskit Function de Qedma
> **Note:** Las Qiskit Functions son una funcionalidad experimental disponible únicamente para usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (a través de la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y están sujetas a cambios.

## Descripción general
Si bien las unidades de procesamiento cuántico han mejorado enormemente en los últimos años, los errores debidos al ruido y las imperfecciones del hardware existente siguen siendo un desafío central para los desarrolladores de algoritmos cuánticos. A medida que el campo se acerca a computaciones cuánticas a escala utilitaria que no pueden verificarse de forma clásica, las soluciones para cancelar el ruido con precisión garantizada adquieren cada vez más importancia. Para superar este desafío, Qedma ha desarrollado Quantum Error Suppression and Error Mitigation (QESEM), integrado de forma transparente en IBM Quantum Platform como una [Qiskit Function](/guides/functions).

Con QESEM, los usuarios pueden ejecutar sus circuitos cuánticos en QPUs ruidosas para obtener resultados altamente precisos y libres de errores con sobrecargas de tiempo de QPU muy eficientes, cercanas a los límites fundamentales. Para lograrlo, QESEM aprovecha un conjunto de métodos propietarios desarrollados por Qedma para la caracterización y reducción de errores. Las técnicas de reducción de errores incluyen optimización de puertas, transpilación consciente del ruido, supresión de errores (ES) y mitigación de errores imparcial (EM). Con esta combinación de métodos basados en caracterización, los usuarios pueden obtener resultados confiables y libres de errores para circuitos cuánticos genéricos de gran volumen, desbloqueando aplicaciones que de otro modo no podrían realizarse.

Para una descripción completa de los componentes subyacentes, así como una demostración a escala utilitaria, consulta el artículo [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Descripción
Puedes usar la función QESEM de Qedma para estimar y ejecutar fácilmente tus circuitos con supresión y mitigación de errores, logrando mayores volúmenes de circuito y mayor precisión. Para usar QESEM, debes proporcionar un circuito cuántico, un conjunto de observables a medir, una precisión estadística objetivo para cada observable y una QPU seleccionada. Antes de ejecutar el circuito con la precisión objetivo, puedes estimar el tiempo de QPU necesario basándote en un cálculo analítico que no requiere ejecución del circuito. Una vez que estés satisfecho con la estimación de tiempo de QPU, puedes ejecutar el circuito con QESEM.

Cuando ejecutas un circuito, QESEM ejecuta un protocolo de caracterización del dispositivo adaptado a tu circuito, produciendo un modelo de ruido confiable para los errores que ocurren en él. A partir de la caracterización, QESEM primero implementa transpilación consciente del ruido para mapear el circuito de entrada a un conjunto de qubits físicos y puertas, minimizando el ruido que afecta al observable objetivo. Esto incluye las puertas disponibles de forma nativa (CX/CZ en dispositivos IBM&reg;), así como puertas adicionales optimizadas por QESEM, formando el conjunto de puertas extendido de QESEM. Luego, QESEM ejecuta un conjunto de circuitos ES y EM basados en caracterización en la QPU y recopila los resultados de sus mediciones. Estos se procesan clásicamente para proporcionar un valor de expectación imparcial y una barra de error para cada observable, correspondiente a la precisión solicitada.

![Descripción general de Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
Se ha demostrado que QESEM proporciona resultados de alta precisión para una variedad de aplicaciones cuánticas y en los mayores volúmenes de circuito alcanzables hoy en día. QESEM ofrece las siguientes características para el usuario, demostradas en la sección de benchmarks a continuación:
-	**Precisión garantizada:** QESEM genera estimaciones imparciales de los valores de expectación de los observables. Su método EM está equipado con garantías teóricas que, junto con la caracterización de vanguardia de Qedma, aseguran que la mitigación converge a la salida del circuito sin ruido hasta la precisión especificada por el usuario. A diferencia de muchos métodos EM heurísticos propensos a errores sistemáticos o sesgos, la precisión garantizada de QESEM es esencial para asegurar resultados confiables en circuitos cuánticos y observables genéricos.
-	**Escalabilidad a QPUs grandes:** El tiempo de QPU de QESEM depende del volumen del circuito, pero en lo demás es independiente del número de qubits. Qedma ha demostrado QESEM en los dispositivos cuánticos más grandes disponibles hoy, incluidos los dispositivos IBM Quantum Eagle de 127 qubits y Heron de 133 qubits.
-	**Agnóstico a la aplicación:** QESEM se ha demostrado en una variedad de aplicaciones, incluyendo simulación hamiltoniana, VQE, QAOA y estimación de amplitud. Los usuarios pueden ingresar cualquier circuito cuántico y observable a medir, y obtener resultados precisos y libres de errores. Las únicas limitaciones las dictan las especificaciones del hardware y el tiempo de QPU asignado, que determinan los volúmenes de circuito accesibles y las precisiones de salida. En contraste, muchas soluciones de reducción de errores son específicas de la aplicación o involucran heurísticas no controladas, lo que las hace inaplicables para circuitos y aplicaciones cuánticos genéricos.
-  **Conjunto de puertas extendido:** QESEM soporta puertas de ángulo fraccionario y proporciona puertas $Rzz(\theta)$ de ángulo fraccionario optimizadas por Qedma en dispositivos IBM Quantum Eagle. Este conjunto de puertas extendido permite una compilación más eficiente y desbloquea volúmenes de circuito mayores por un factor de hasta 2 en comparación con la compilación predeterminada de CX/CZ.
-	**Observables multibase:** QESEM soporta observables de entrada compuestos de muchas cadenas de Pauli no conmutantes, como Hamiltonianos genéricos. La elección de las bases de medición y la optimización de la asignación de recursos de QPU (shots y circuitos) se realiza automáticamente por QESEM para minimizar el tiempo de QPU requerido para la precisión solicitada. Esta optimización, que tiene en cuenta las fidelidades del hardware y las tasas de ejecución, te permite ejecutar circuitos más profundos y obtener mayor precisión.
## Benchmarks
QESEM ha sido probado en una amplia variedad de casos de uso y aplicaciones. Los siguientes ejemplos pueden ayudarte a evaluar qué tipos de cargas de trabajo puedes ejecutar con QESEM.

Una figura de mérito clave para cuantificar la dificultad tanto de la mitigación de errores como de la simulación clásica para un circuito y observable dado es el **volumen activo**: el número de puertas CNOT que afectan al observable en el circuito. El volumen activo depende de la profundidad y el ancho del circuito, del peso del observable y de la estructura del circuito, que determina el cono de luz del observable. Para más detalles, consulta la charla del [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM proporciona un valor particularmente grande en el régimen de alto volumen, ofreciendo resultados confiables para circuitos y observables genéricos.

![Volumen activo](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplicación    | Número de qubits | Dispositivo | Descripción del circuito | Precisión | Tiempo total | Uso de runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuito VQE  | 8              | Eagle (r3) | 21 capas totales, 9 bases de medición, cadena 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 capas únicas x 3 pasos, topología heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 capas únicas x 8 pasos, topología heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulación hamiltoniana Trotterizada   | 40  | Eagle (r3)            | 2 capas únicas x 10 pasos de Trotter, cadena 1D                    | 97%      | 3 horas     | 25 min         |
| Simulación hamiltoniana Trotterizada   | 119  | Eagle (r3)           | 3 capas únicas x 9 pasos de Trotter, topología heavy-hex 2D                    | 95%      | 6,5 horas     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 capas únicas x 15 pasos, topología heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

La precisión se mide aquí en relación con el valor ideal del observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, donde '$\epsilon$' es la precisión absoluta de la mitigación (establecida por el usuario), y $\langle O \rangle_{ideal}$ es el observable del circuito sin ruido.
El «uso de runtime» mide el uso del benchmark en modo batch (suma sobre el uso de los trabajos individuales), mientras que el «tiempo total» mide el uso en modo sesión (tiempo de pared del experimento), que incluye tiempos adicionales de comunicación y procesamiento clásico. QESEM está disponible para ejecución en ambos modos, de modo que los usuarios puedan aprovechar al máximo sus recursos disponibles.

Los circuitos Kicked Ising de 28 qubits simulan el Cristal de Tiempo Discreto estudiado por Shinjo et al. (ver [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) y [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) en tres bucles conectados de ibm_kawasaki. Los parámetros del circuito tomados aquí son $(\theta_x, \theta_z) = (0.9 \pi, 0)$, con un estado inicial ferromagnético $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. El observable medido es el valor absoluto de la magnetización $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. El experimento Kicked Ising a escala utilitaria se ejecutó en los 136 mejores qubits de ibm_fez; este benchmark particular se ejecutó al ángulo de Clifford $(\theta_x, \theta_z) = (\pi, 0)$, en el que el volumen activo crece lentamente con la profundidad del circuito, lo que, junto con las altas fidelidades del dispositivo, permite alta precisión en un tiempo de ejecución corto.

Los circuitos de simulación hamiltoniana Trotterizada corresponden a un modelo de Ising de campo transverso con ángulos fraccionarios: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ y $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ respectivamente (ver [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). El circuito a escala utilitaria se ejecutó en los 119 mejores qubits de ibm_brisbane, mientras que el experimento de 40 qubits se ejecutó en la mejor cadena disponible. La precisión se reporta para la magnetización; también se obtuvieron resultados de alta precisión para observables de mayor peso.

El circuito VQE fue desarrollado junto con investigadores del Center for Quantum Technology and Applications del Deutsches Elektronen-Synchrotron (DESY). El observable objetivo aquí fue un Hamiltoniano compuesto por un gran número de cadenas de Pauli no conmutantes, destacando el rendimiento optimizado de QESEM para observables multibases. La mitigación se aplicó a un ansatz optimizado clásicamente; aunque estos resultados aún no están publicados, se obtendrán resultados de la misma calidad para diferentes circuitos con propiedades estructurales similares.
## Comenzar
Autentícate usando tu [clave de API de IBM Quantum Platform](http://quantum.cloud.ibm.com/), y selecciona la Qiskit Function QESEM de la siguiente manera. (Este fragmento asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Ejemplo
Para comenzar, prueba este ejemplo básico de estimación del tiempo de QPU necesario para ejecutar QESEM con un `pub` dado:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

El siguiente ejemplo ejecuta un trabajo QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Puedes usar las APIs familiares de Qiskit Serverless para verificar el estado de tu carga de trabajo de Qiskit Function o recuperar los resultados:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Parámetros de la función
| Nombre |  Tipo | Descripción | Requerido | Predeterminado |  Ejemplo |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) |Esta es la entrada principal. El `Pub` contiene de 2 a 4 elementos: un circuito, uno o más observables, 0 o un único conjunto de valores de parámetros y una precisión opcional. Si no se especificó una precisión, se usará la `default_precision` de `options`|  Sí|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string|Nombre del backend a usar |No | QESEM seleccionará el dispositivo menos ocupado reportado por IBM| `"ibm_fez"`|
| `instance` | string|  El nombre de recurso en la nube de la instancia a usar en ese formato |  No |  N/A | `"CRN"`  |
| `options` | dictionary |Opciones de entrada. Consulta la sección **Options** para más detalles. | No |  Consulta la sección **Options** para los detalles.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### Options
| Opción |  Valores | Descripción | Predeterminado |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | Cuando se establece, el trabajo QESEM solo calculará la estimación del tiempo de QPU. Consulta la descripción a continuación para más detalles. Cuando se establece en None, el circuito se ejecutará con QESEM| None |
|`default_precision` | 0 < float | Se aplicará a los `pubs` que no tengan precisión especificada. La precisión indica el error aceptable en los valores de expectación de los observables en valor absoluto. Es decir, el tiempo de ejecución de QPU para la mitigación se determinará para proporcionar valores de salida para todos los observables de interés que caigan dentro de un intervalo de confianza `1σ` de la precisión objetivo. Si se proporcionan múltiples observables, la mitigación se ejecutará hasta que se alcance la precisión objetivo para cada uno de los observables de entrada. | 0.02|
|`max_execution_time` | 0 < integer < 28,800 (8 horas)| Te permite limitar el tiempo de QPU, especificado en segundos, a utilizar para todo el proceso QESEM. Consulta los detalles adicionales a continuación.| 3,600 (una hora)|
| `transpilation_level` | 0 / 1 | Consulta la descripción a continuación | 1|
| `execution_mode` | `"session"` / `"batch"` | Consulta la descripción a continuación | "batch"|

> **Caution:** La estimación del tiempo de QPU cambia de un backend a otro. Por lo tanto, al ejecutar la función QESEM, asegúrate de ejecutarla en el mismo backend que se selecciónó al obtener la estimación de tiempo de QPU.

> **Note:** QESEM terminará su ejecución cuando alcance la precisión objetivo o cuando alcance el `max_execution_time`, lo que ocurra primero.

- `estimate_time_only` - Esta bandera permite a los usuarios obtener una estimación del tiempo de QPU necesario para ejecutar el circuito con QESEM.
    - Si se establece en `"analytical"`, se calcula una cota superior del tiempo de QPU sin consumir ningún uso de QPU. Esta estimación tiene una resolución de 30 minutos (por ejemplo, 30 minutos, 60 minutos, 90 minutos, etc.). Generalmente es pesimista y solo puede obtenerse para observables de Pauli simples o sumas de Paulis sin soportes que se intersecten (por ejemplo, Z0+Z1). Es principalmente útil para comparar los niveles de complejidad de los diferentes parámetros proporcionados por el usuario (circuito, precisión, etc.).
    - Para obtener una estimación del tiempo de QPU más precisa, establece esta bandera en `"empirical"`. Aunque esta opción requiere ejecutar un pequeño número de circuitos, proporciona una estimación del tiempo de QPU significativamente más precisa. Esta estimación tiene una resolución de 5 minutos (por ejemplo, 20 minutos, 25 minutos, 30 minutos, etc.). El usuario puede elegir ejecutar la estimación de tiempo empírica en modo batch o sesión. Para más detalles, consulta la descripción de `execution_mode`. Por ejemplo, en modo batch, la estimación de tiempo empírica consumirá menos de 10 minutos de tiempo de QPU.

- `max_execution_time`: Te permite limitar el tiempo de QPU, especificado en segundos, a utilizar para todo el proceso QESEM. Dado que el tiempo de QPU final requerido para alcanzar la precisión objetivo se determina dinámicamente durante el trabajo QESEM, este parámetro te permite limitar el costo del experimento. Si el tiempo de QPU determinado dinámicamente es menor que el tiempo asignado por el usuario, este parámetro no afectará el experimento. El parámetro `max_execution_time` es particularmente útil en casos donde la estimación de tiempo analítica proporcionada por QESEM antes de que comience el trabajo es demasiado pesimista y el usuario desea iniciar un trabajo de mitigación de todos modos. Una vez alcanzado el límite de tiempo, QESEM deja de enviar nuevos circuitos. Los circuitos que ya se han enviado continúan ejecutándose (por lo que el tiempo total puede superar el límite en hasta 30 minutos), y el usuario recibe los resultados procesados de los circuitos que se ejecutaron hasta ese momento. Si deseas aplicar un límite de tiempo de QPU menor que la estimación de tiempo analítica, consulta con Qedma para obtener una estimación de la precisión alcanzable dentro del límite de tiempo.

- `transpilation_level`: Después de que se envía un circuito a QESEM, este prepara automáticamente varias transpilaciones alternativas del circuito y elige la que minimiza el tiempo de QPU. Por ejemplo, las transpilaciones alternativas pueden utilizar puertas RZZ fraccionarias optimizadas por Qedma para reducir la profundidad del circuito. Por supuesto, todas las transpilaciones son equivalentes al circuito de entrada en términos de su salida ideal. Para ejercer mayor control sobre la transpilación del circuito, establece el nivel de transpilación en `options`. Mientras que `"transpilation_level": 1` corresponde al comportamiento predeterminado descrito anteriormente, `"transpilation_level": 0` incluye solo las modificaciones mínimas necesarias al circuito original; por ejemplo, la «layerificación», es decir, la organización de las operaciones del circuito en «capas» de puertas de dos qubits simultáneas. Ten en cuenta que el mapeo automático de hardware a qubits de alta fidelidad se aplica en cualquier caso.

| transpilation_level | Descripción |
|:-:|:--|
| `1` | Transpilación QESEM predeterminada. Prepara varias transpilaciones alternativas y elige la que minimiza el tiempo de QPU. Las barreras pueden modificarse en el paso de layerificación. |
| `0` | Transpilación mínima: el circuito mitigado se parecerá estructuralmente al circuito de entrada. Los circuitos proporcionados en el nivel 0 deben coincidir con la conectividad del dispositivo y especificarse en términos de las siguientes puertas: CX, Rzz(α) y puertas estándar de un solo qubit (U, x, sx, rz, etc.). Las barreras se respetarán en el paso de layerificación. |

- `execution_mode` - El usuario puede elegir ejecutar el trabajo QESEM en una [sesión IBM](/guides/execution-modes#session-mode) dedicada o a través de múltiples [batches IBM](/guides/execution-modes#batch-mode):
    -   **Modo sesión**: Las sesiones son más costosas pero resultan en un tiempo de resultado más corto. Una vez que comienza la sesión, la QPU se reserva exclusivamente para el trabajo QESEM. El cálculo de uso incluye tanto el tiempo dedicado a la ejecución en la QPU como las computaciones clásicas asociadas (realizadas por QESEM e IBM). La Qiskit Function QESEM se encarga de crear y cerrar la sesión automáticamente. Para usuarios con acceso ilimitado a las QPUs (por ejemplo, configuraciones on-premises), se recomienda usar el modo sesión para una ejecución más rápida de QESEM.
    -   **Modo batch**: En modo batch, la QPU se libera durante las computaciones clásicas, lo que conlleva un menor uso de QPU. Como los trabajos en batch típicamente abarcan un período más largo, existe un mayor riesgo de derivas del hardware; QESEM incorpora medidas para detectar y compensar las derivas, manteniendo la confiabilidad durante ejecuciones prolongadas.

> **Note:** Las operaciones de barrera se usan típicamente para especificar las capas de puertas de dos qubits en circuitos cuánticos. En el nivel 0, QESEM preserva las capas especificadas por las barreras. En el nivel 1, las capas especificadas por las barreras se consideran como una alternativa de transpilación al minimizar el tiempo de QPU.
### Salidas
La salida de una función de circuito es un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), que contiene dos campos:

- Un objeto [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Se puede indexar directamente desde el `PrimitiveResult`.

- Metadatos a nivel de trabajo.

Cada `PubResult` contiene un campo `data` y un campo `metadata`.

- El campo `data` contiene al menos un arreglo de valores de expectación (`PubResult.data.evs`) y un arreglo de errores estándar (`PubResult.data.stds`). También puede contener más datos, dependiendo de las opciones usadas.

- El campo `metadata` contiene metadatos a nivel de PUB (`PubResult.metadata`).

El siguiente fragmento de código describe cómo recuperar la estimación del tiempo de QPU (cuando `estimate_time_only` está establecido):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

El siguiente fragmento de código demuestra cómo recuperar los resultados de mitigación (cuando `estimate_time_only` no está establecido) y las métricas de ejecución. Estos contienen datos esenciales que permiten una comprensión más profunda de cómo los diferentes parámetros impactan la ejecución de QESEM. También puede ser relevante al escribir un artículo basado en tu investigación.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Obtener mensajes de error
Si el estado de tu carga de trabajo es ERROR, usa job.result() para recuperar el mensaje de error de la siguiente manera:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>